# Import Dependencies

In [1]:
import pandas as pd
import logging
from pathlib import Path
from tools import cleaner
from scripts import downloader

/home/hwangwy/Projects/Python/Assignments/Ecommerce-Analyst/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Ingest Dataset

In [3]:
raw_df_ls = {}

try:
    url = "erfan4524/e-commerce-sales-data-analysis-and-eda"
    data_path = 'data/'
    downloader.dataset_download(url, output_dir=data_path)

    for file in Path(data_path).iterdir():
        file_name = Path(file).name
        if file_name == ".complete" or file_name == "clean_final_data.csv":
            continue
        else:
            logger.info(f"CSV file founded: {file_name}")
            raw_df_ls[file_name] = pd.read_csv(f"{data_path}/{file_name}")
            logger.info(f"Added into df_ls")
except Exception as e:
    print(e)

2026-09-14 21:08:57,317 - scripts.downloader - INFO - Dataset is already downloaded at data/
2026-09-14 21:08:57,318 - __main__ - INFO - CSV file founded: customers.csv
2026-09-14 21:08:57,333 - __main__ - INFO - Added into df_ls
2026-09-14 21:08:57,334 - __main__ - INFO - CSV file founded: orders.csv
2026-09-14 21:08:57,372 - __main__ - INFO - Added into df_ls
2026-09-14 21:08:57,372 - __main__ - INFO - CSV file founded: payments.csv
2026-09-14 21:08:57,392 - __main__ - INFO - Added into df_ls
2026-09-14 21:08:57,393 - __main__ - INFO - CSV file founded: products.csv
2026-09-14 21:08:57,394 - __main__ - INFO - Added into df_ls


# Dataset Information

These block of codes will show the data inside csv files and show what's need to clean

# Customers.csv

In [4]:
df: pd.DataFrame = raw_df_ls['customers.csv']
cleaner.view_data(df)
print(df[(df['Age'].isna() & df['City'].isna())])
print(df['CustomerSegment'].unique())

===========HEAD==============
   CustomerID   Age    City  SignupDate CustomerSegment
0      100001  22.0  Tehran  2023-09-11         Regular
1      100002  55.0  Tabriz  2024-01-16             VIP
2      100003  49.0   Karaj  2025-07-31             New
3      100004  39.0  Tehran  2023-04-23             New
4      100005  38.0  Tehran  2023-04-29         Regular
===========SHAPE==============
Dimension: (10000, 5)
===========DESCRIBE==============
          CustomerID          Age    City  SignupDate CustomerSegment
count    10000.00000  9820.000000    9881       10000           10000
unique           NaN          NaN      12        1094               3
top              NaN          NaN  Tehran  2023-01-07         Regular
freq             NaN          NaN    2733          21            5563
mean    105000.50000    41.321487     NaN         NaN             NaN
std       2886.89568    13.802441     NaN         NaN             NaN
min     100001.00000    18.000000     NaN         NaN    

First look into the dataset, we will se Age column is float64 instead of int64, SignupDate is str instead of date. Secondly is handle null values, in the Age column instead of dropping Age, we will fill it with mean values and the change the datatype into int64. For City column because it contains 12 unique values so we drop it all (Note that these two column have a total of 1000 rows so you can't drop 300+ rows of both values. This will affect the quality of the dataset)

Run the pipeline in the cleaner tools

In [5]:
df = cleaner.clean_customer(df)
cleaner.view_data(df)
print(df[df['City'].isna()])

2026-09-14 21:08:57,446 - tools.cleaner - INFO - Columns formatted
2026-09-14 21:08:57,451 - tools.cleaner - INFO - Dropped duplicates
2026-09-14 21:08:57,452 - tools.cleaner - INFO - Filled Age Null values to mean
2026-09-14 21:08:57,453 - tools.cleaner - INFO - Changed Age dtype to int
2026-09-14 21:08:57,455 - tools.cleaner - INFO - Dropped Null values from City
2026-09-14 21:08:57,460 - tools.cleaner - INFO - Formatted SignupDate to type datetime


===========HEAD==============
   CustomerID  Age    City SignupDate CustomerSegment
0      100001   22  Tehran 2023-09-11         Regular
1      100002   55  Tabriz 2024-01-16             VIP
2      100003   49   Karaj 2025-07-31             New
3      100004   39  Tehran 2023-04-23             New
4      100005   38  Tehran 2023-04-29         Regular
===========SHAPE==============
Dimension: (9881, 5)
===========DESCRIBE==============
           CustomerID          Age    City                  SignupDate  \
count     9881.000000  9881.000000    9881                        9881   
unique            NaN          NaN      12                         NaN   
top               NaN          NaN  Tehran                         NaN   
freq              NaN          NaN    2733                         NaN   
mean    104994.432345    41.314543     NaN  2024-06-30 20:16:44.108895   
min     100001.000000    18.000000     NaN         2023-01-01 00:00:00   
25%     102492.000000    30.000000     NaN

# Orders.csv

In [6]:
df: pd.DataFrame = raw_df_ls['orders.csv']
cleaner.view_data(df)
print(df[df['Quantity'].isna()].head(5))
print(df['PaymentMethod'].unique())
print(df[df['PaymentMethod'].isna()].head(5))
print(df['Status'].unique())

===========HEAD==============
   OrderID  CustomerID   OrderDate  ProductID  Quantity  Discount  \
0   500001      103695  2025-08-28       2003       4.0       0.1   
1   500002      107935  2024-05-31       2014       1.0       0.1   
2   500003      105141  2025-08-10       2002       1.0       0.2   
3   500004      108780  2024-10-11       2012       1.0       0.1   
4   500005      101187  2024-02-19       2008       2.0       0.0   

  PaymentMethod     Status  
0       Gateway  Completed  
1        Wallet  Completed  
2    CardToCard  Completed  
3       Gateway  Completed  
4       Gateway  Completed  
===========SHAPE==============
Dimension: (50120, 8)
===========DESCRIBE==============
              OrderID     CustomerID   OrderDate     ProductID      Quantity  \
count    50120.000000   50120.000000       50085  50120.000000  50040.000000   
unique            NaN            NaN         912           NaN           NaN   
top               NaN            NaN  2025-09-05      

In the orders.csv file, we will see that the OrderDate have the same issue with customers.csv which datatype is string instead of date, Quantity is float64 and have some outlier values that < 0. Discount is 0.x format which is hard to read. In this file, there are four columns with null values, we should drop OrderDate because dropping it won't affect much the dataset which has 50k of rows dropping 35 rows is not a big deal. For quantity columns, if we fill these values with a single value or mean it will go wrong. We should drop these values instead of filling it. The discount can be filled with 0 value instead of dropping it. Lastly, PaymentMethod can just be filled with Cash

In [7]:
df = cleaner.clean_orders(df)
cleaner.view_data(df)

2026-09-14 21:08:57,529 - tools.cleaner - INFO - Columns formatted
2026-09-14 21:08:57,544 - tools.cleaner - INFO - Dropped duplicates
2026-09-14 21:08:57,549 - tools.cleaner - INFO - Dropped OrderDate Null values
2026-09-14 21:08:57,560 - tools.cleaner - INFO - Formatted OrderDate dtype to date
2026-09-14 21:08:57,562 - tools.cleaner - INFO - Filled Discount Null values with 0
2026-09-14 21:08:57,581 - tools.cleaner - INFO - Formatted Discount to percentage value
2026-09-14 21:08:57,585 - tools.cleaner - INFO - Changed dtype from float to int for Quantity column
2026-09-14 21:08:57,589 - tools.cleaner - INFO - Dropped outlier values in Quantity column
2026-09-14 21:08:57,591 - tools.cleaner - INFO - Filled Null value from PaymentMethod to Cash


===========HEAD==============
   OrderID  CustomerID  OrderDate  ProductID  Quantity Discount PaymentMethod  \
0   500001      103695 2025-08-28       2003         4    10.0%       Gateway   
1   500002      107935 2024-05-31       2014         1    10.0%        Wallet   
2   500003      105141 2025-08-10       2002         1    20.0%    CardToCard   
3   500004      108780 2024-10-11       2012         1    10.0%       Gateway   
4   500005      101187 2024-02-19       2008         2     0.0%       Gateway   

      Status  
0  Completed  
1  Completed  
2  Completed  
3  Completed  
4  Completed  
===========SHAPE==============
Dimension: (49860, 8)
===========DESCRIBE==============
              OrderID     CustomerID                   OrderDate  \
count    49860.000000   49860.000000                       49860   
unique            NaN            NaN                         NaN   
top               NaN            NaN                         NaN   
freq              NaN            N

Results after running the pipeline

# Payments.csv

In [8]:
df = raw_df_ls["payments.csv"]
cleaner.view_data(df)

===========HEAD==============
   PaymentID  OrderID PaymentDate PaymentStatus
0     800001   500001  2025-08-28          Paid
1     800002   500002  2024-05-31          Paid
2     800003   500003  2025-08-10          Paid
3     800004   500004  2024-10-11          Paid
4     800005   500005  2024-02-19          Paid
===========SHAPE==============
Dimension: (50000, 4)
===========DESCRIBE==============
            PaymentID        OrderID PaymentDate PaymentStatus
count    50000.000000   50000.000000       49965         50000
unique            NaN            NaN         912             3
top               NaN            NaN  2025-09-05          Paid
freq              NaN            NaN          88         46569
mean    825000.500000  525000.500000         NaN           NaN
std      14433.901067   14433.901067         NaN           NaN
min     800001.000000  500001.000000         NaN           NaN
25%     812500.750000  512500.750000         NaN           NaN
50%     825000.500000  52500

This dataset has the same problems that PaymentDate is str and contains null. Because it only has 35 null values so we will drop them.

In [9]:
df = cleaner.clean_payments(df)
cleaner.view_data(df)

2026-09-14 21:08:57,658 - tools.cleaner - INFO - Columns formatted
2026-09-14 21:08:57,664 - tools.cleaner - INFO - Dropped duplicates
2026-09-14 21:08:57,668 - tools.cleaner - INFO - Dropped Null values from PaymentDate
2026-09-14 21:08:57,677 - tools.cleaner - INFO - Changed PaymentDate to type date


===========HEAD==============
   PaymentID  OrderID PaymentDate PaymentStatus
0     800001   500001  2025-08-28          Paid
1     800002   500002  2024-05-31          Paid
2     800003   500003  2025-08-10          Paid
3     800004   500004  2024-10-11          Paid
4     800005   500005  2024-02-19          Paid
===========SHAPE==============
Dimension: (49965, 4)
===========DESCRIBE==============
            PaymentID        OrderID                 PaymentDate PaymentStatus
count    49965.000000   49965.000000                       49965         49965
unique            NaN            NaN                         NaN             3
top               NaN            NaN                         NaN          Paid
freq              NaN            NaN                         NaN         46534
mean    825000.973521  525000.973521  2025-03-11 18:05:44.545181           NaN
min     800001.000000  500001.000000         2024-01-01 00:00:00           NaN
25%     812501.000000  512501.000000      

In [10]:
df = raw_df_ls['products.csv']
cleaner.view_data(df)

===========HEAD==============
   ProductID          ProductName     Category  UnitPrice
0       2001       Wireless Mouse  Electronics       18.0
1       2002  Mechanical Keyboard  Electronics       62.0
2       2003          USB-C Cable  Accessories        9.0
3       2004           Power Bank  Electronics       35.0
4       2005         Laptop Stand  Accessories       28.0
===========SHAPE==============
Dimension: (20, 4)
===========DESCRIBE==============
         ProductID     ProductName     Category   UnitPrice
count     20.00000              20           20   20.000000
unique         NaN              20            6         NaN
top            NaN  Wireless Mouse  Electronics         NaN
freq           NaN               1            9         NaN
mean    2010.50000             NaN          NaN   61.400000
std        5.91608             NaN          NaN   62.836798
min     2001.00000             NaN          NaN    7.000000
25%     2005.75000             NaN          NaN   20.25000

This file does not contain anything to clean.